In [1]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import opt_einsum as oe
from dataclasses import dataclass
from typing import List

In [2]:
in_shape: t.Size = t.Size([8, 8, 8])
out_shape: t.Size = t.Size([16, 16, 16])
rank: t.Size = t.Size([1, 4, 4, 1])
N = len(in_shape)

In [ ]:
cores = []
for i in range(N):
    G = t.randn(rank[i], in_shape[i], out_shape[i], rank[i+1])
    cores.append(G)

In [ ]:
for c in cores:
    print(c.shape)

torch.Size([1, 8, 16, 4])
torch.Size([4, 8, 16, 4])
torch.Size([4, 8, 16, 1])


In [59]:
X = t.randn(t.Size([32]) + in_shape)

In [60]:
def build_expr(X_shape, *cores_shape):
    k = 0
    X_expr = Y_expr = oe.get_symbol(k)
    k += 1
    cores_expr = []
    for n in range(len(cores_shape)):
        r_minus = oe.get_symbol(k)
        i_n = oe.get_symbol(k+1)
        j_n = oe.get_symbol(k+2)
        r_plus = oe.get_symbol(k+3)
        G_expr = r_minus + i_n + j_n + r_plus
        cores_expr.append(G_expr)
        X_expr += i_n
        Y_expr += j_n
        k += 3
    expr = X_expr + "," + ','.join(cores_expr) + "->" + Y_expr
    return expr

In [61]:
expr = build_expr(t.Size([]), *[c.shape for c in cores])

In [62]:
path, path_info = oe.contract_path(
    expr, X.shape, *[c.shape for c in cores], shapes=True)

In [63]:
steps = [(c[0], c[2]) for c in path_info.contraction_list]

In [64]:
steps

[((1, 0), 'bcde,acfi->deafi'),
 ((2, 0), 'deafi,efgh->daigh'),
 ((1, 0), 'daigh,hijk->adgj')]

In [65]:
operands = [X, *cores]
for positions, einsum_str in steps:
    print(positions, einsum_str)
    a, b = operands[positions[0]], operands[positions[1]]
    res = t.einsum(einsum_str, a, b)
    operands = [op for i, op in enumerate(
        operands) if i not in positions] + [res]

(1, 0) bcde,acfi->deafi
(2, 0) deafi,efgh->daigh
(1, 0) daigh,hijk->adgj


In [68]:
res.shape

torch.Size([32, 16, 16, 16])

In [31]:
def ttm_2_matrix(cores: List[t.Tensor]):
    G = cores[0]
    for i in range(1, N):
        G = t.tensordot(G, cores[i], dims=([-1], [0]))
    G = G.squeeze()
    G = t.permute(G, [i for i in range(2*N) if i % 2 == 0] +
                  [i for i in range(2*N) if i % 2 != 0])
    G = G.reshape(G.shape[:N].numel(), G.shape[N:].numel())
    return G

In [ ]:
def get_contr_expr(X_shape: t.Size, *cores_shape: t.Size):
    k = 0
    X_expr = Y_expr = oe.get_symbol(k)
    k += 1
    cores_expr = []
    for n in range(len(cores_shape)):
        r_minus = oe.get_symbol(k)
        i_n = oe.get_symbol(k+1)
        j_n = oe.get_symbol(k+2)
        r_plus = oe.get_symbol(k+3)
        G_expr = r_minus + i_n + j_n + r_plus
        cores_expr.append(G_expr)
        X_expr += i_n
        Y_expr += j_n
        k += 3
    expr = X_expr + "," + ','.join(cores_expr) + "->" + Y_expr
    return oe.contract_expression(expr, X_shape, *cores_shape)

In [ ]:
def ttm_matvec(contr_expr, X: t.Tensor, *cores: t.Tensor):
    return contr_expr(X, *cores, backend='torch')

In [17]:
def get_ttm_std(rank: t.Size, std):
    # TODO: make proper initialization
    return std


def init_ttm_cores(in_shape: t.Size, out_shape: t.Size, rank: t.Size, std):
    cores = []
    N = len(in_shape)
    for i in range(N):
        G = std * t.randn(rank[i], in_shape[i], out_shape[i], rank[i+1])
        cores.append(G)
    return cores

In [ ]:
@dataclass
class TTMLinearConfig:
    in_shape: t.Size
    out_shape: t.Size
    rank: t.Size
    seq_len: int
    batch_size: int
    init_std: float = 0.02


class TTMLinear(nn.Module):
    def __init__(self, cfg: TTMLinearConfig):
        super().__init__()
        self.cfg = cfg
        self.cores = self._build_cores()
        self.steps = self.get_contr_expr(self.cfg.seq_len, self.cfg.batch_size)

    def _build_cores(self) -> List[nn.Parameter]:
        rank = self.cfg.rank
        in_shape = self.cfg.in_shape
        out_shape = self.cfg.out_shape
        std = get_ttm_std(rank, self.cfg.init_std)
        res = [nn.Parameter(core) for core in init_ttm_cores(
            in_shape, out_shape, rank, std)]
        return res

    def _build_expr(self, *cores_shape: t.Size):
        k = 0
        X_expr = Y_expr = oe.get_symbol(k)
        k += 1
        cores_expr = []
        for n in range(len(cores_shape)):
            r_minus = oe.get_symbol(k)
            i_n = oe.get_symbol(k+1)
            j_n = oe.get_symbol(k+2)
            r_plus = oe.get_symbol(k+3)
            G_expr = r_minus + i_n + j_n + r_plus
            cores_expr.append(G_expr)
            X_expr += i_n
            Y_expr += j_n
            k += 3
        expr = X_expr + "," + ','.join(cores_expr) + "->" + Y_expr
        return expr

    def get_cores(self, masked: bool = True) -> List[t.Tensor]:
        # TODO: add masking
        return list(self.cores)

    def get_contr_expr(self, seq_len, batch_size):
        self.cfg.seq_len, self.cfg.batch_size = seq_len, batch_size
        X_shape = t.Size([seq_len * batch_size]) + self.cfg.in_shape
        cores_shape = [c.shape for c in self.get_cores()]
        expr = self._build_expr(*cores_shape)
        _, path_info = oe.contract_path(
            expr, X_shape, *cores_shape, shapes=True)
        steps = [(c[0], c[2]) for c in path_info.contraction_list]
        return steps

    def forward(self, X):
        X_ten = X.reshape(self.cfg.seq_len *
                          self.cfg.batch_size, *self.cfg.in_shape)
        operands = [X_ten, *self.get_cores()]
        for positions, einsum_str in self.steps:
            a, b = operands[positions[0]], operands[positions[1]]
            Y = t.einsum(einsum_str, a, b)
            operands = [op for i, op in enumerate(
                operands) if i not in positions] + [Y]
        return Y.reshape(self.cfg.seq_len, self.cfg.batch_size, *self.cfg.out_shape)

In [97]:
cfg = TTMLinearConfig(
    in_shape=t.Size([8, 8, 8]),
    out_shape=t.Size([16, 16, 16]),
    rank=t.Size([1, 4, 4, 1]),
    seq_len=128,
    batch_size=32
)

In [98]:
layer = TTMLinear(cfg)
layer_comp = t.compile(layer, fullgraph=True)

In [102]:
X = t.randn(t.Size([128]) + t.Size([32]) + cfg.in_shape)
X_mat = X.reshape(-1, cfg.in_shape.numel())

In [103]:
Y = layer(X)

In [104]:
W_mat = ttm_2_matrix(layer.get_cores(masked=False))
Y_mat = (X_mat @ W_mat).reshape(X.shape[0], X.shape[1], -1)

In [105]:
l = t.linalg.norm(Y)

In [106]:
l.backward()

In [107]:
for _ in range(10):
    layer_comp(X)

In [110]:
%timeit layer(X)

45.8 ms ± 3.81 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [111]:
%timeit layer_comp(X)

37 ms ± 3.25 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
